# Silver — GEO (ficheiros SOFT → bronze family)

Consolida **todos** os `data/bronze/bronze_*_family.csv` num único `data/silver/silver_geo.csv`: limpa cada ficheiro com `clean_geo_dataset` (como na task `silver_transform`), junta com `pandas.concat`, deduplica por `sample_id` e mantém a coluna `bronze_source_file` para rastreio.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "dags" / "utils" / "data_cleaners.py").is_file():
            return p
    raise RuntimeError("Abra o notebook a partir da pasta notebooks/ ou da raiz do repositório.")


ROOT = repo_root()
sys.path.insert(0, str(ROOT / "dags"))

from utils.data_cleaners import clean_geo_dataset

bronze_dir = ROOT / "data" / "bronze"
silver_dir = ROOT / "data" / "silver"
SILVER_OUT = silver_dir / "silver_geo.csv"
geo_files = sorted(bronze_dir.glob("bronze_*_family.csv"))
ROOT, geo_files

In [ ]:
if not geo_files:
    raise FileNotFoundError(f"Nenhum bronze_*_family.csv em {bronze_dir}")

parts: list[pd.DataFrame] = []
for file in geo_files:
    raw = pd.read_csv(file)
    cleaned = clean_geo_dataset(raw)
    if cleaned.empty:
        raise ValueError(f"Arquivo sem linhas validas apos limpeza: {file.name}")
    cleaned = cleaned.copy()
    cleaned["bronze_source_file"] = file.name
    parts.append(cleaned)

df_consolidated = pd.concat(parts, ignore_index=True)
if "sample_id" in df_consolidated.columns:
    df_consolidated = df_consolidated.drop_duplicates(subset=["sample_id"], keep="first")

len(geo_files), df_consolidated.shape

In [ ]:
df_consolidated["bronze_source_file"].value_counts()

In [ ]:
df_consolidated.head()

In [ ]:
df_consolidated[
    [c for c in ("sample_id", "geo_accession", "bronze_source_file", "is_blood_sample") if c in df_consolidated.columns]
].head(12)

In [ ]:
# Descomente para gravar o mesmo ficheiro consolidado que a DAG silver escreve.
# silver_dir.mkdir(parents=True, exist_ok=True)
# df_consolidated.to_csv(SILVER_OUT, index=False)
# SILVER_OUT